In [ ]:
import os
import re
import string
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

RANDOM_STATE = 42   # fixed so the whole team gets the identical split
TEST_SIZE = 0.2
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
DATA_PATH = "US-Economic-News.csv"   # matches whatever you just uploaded

df = pd.read_csv(DATA_PATH, encoding="latin-1")
print(f"Loaded dataset with shape {df.shape}")

text_priority = ["text", "article", "body", "content", "headline"]
lower_cols = {c.lower(): c for c in df.columns}
text_col = next((lower_cols[name] for name in text_priority if name in lower_cols), None)

target_priority = ["relevance", "relevant", "class", "label", "target"]
target_col = next((lower_cols[name] for name in target_priority if name in lower_cols), None)

print(f"Text column detected as: '{text_col}' | Target column detected as: '{target_col}'")

df = df[[text_col, target_col]].dropna()
df.columns = ["text", "relevance_raw"]

Loaded dataset with shape (8000, 15)
Text column detected as: 'text' | Target column detected as: 'relevance'


In [ ]:
def norm_label(v):
    v = str(v).strip().lower()
    if v in ("yes", "relevant", "1", "true"):
        return "Relevant"
    return "Not Relevant"

df["label"] = df["relevance_raw"].apply(norm_label)

n_not_sure = df["relevance_raw"].astype(str).str.strip().str.lower().eq("not sure").sum()
if n_not_sure:
    print(f"[NOTE] {n_not_sure} rows had relevance='not sure' and were folded into 'Not Relevant'.")

df = df.drop(columns=["relevance_raw"]).reset_index(drop=True)

print("\nClass distribution (raw, full dataset):")
print(df["label"].value_counts())

[NOTE] 9 rows had relevance='not sure' and were folded into 'Not Relevant'.

Class distribution (raw, full dataset):
label
Not Relevant    6580
Relevant        1420
Name: count, dtype: int64


In [ ]:
STOPWORDS = ENGLISH_STOP_WORDS
PUNCT_TABLE = str.maketrans("", "", string.punctuation)

def preprocess_text(text):
    text = str(text).lower()                       # lowercasing
    text = re.sub(r"</?[a-z]+\s*/?>", " ", text)    # strip HTML tags (e.g. </br>)
    text = text.translate(PUNCT_TABLE)              # punctuation removal
    text = re.sub(r"\d+", " ", text)                # digit removal
    text = re.sub(r"[^a-z\s]", " ", text)           # unnecessary/special characters
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]  # stop-word removal
    text = " ".join(tokens)
    text = re.sub(r"\s+", " ", text).strip()        # whitespace cleanup
    return text

In [ ]:
df["clean_text"] = df["text"].apply(preprocess_text)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

print("Sample cleaned text:")
print(df["clean_text"].iloc[0][:300])

Sample cleaned text:
new york yields certificates deposit offered major banks dropped tenth percentage point latest week reflecting overall decline shortterm rates smalldenomination consumer cds sold directly banks average yield sixmonth deposits fell week ended yesterday according bank survey banxquote money markets wi


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["label"]
)

print(f"Training instances : {len(X_train)}")
print(f"Testing instances  : {len(X_test)}")
print("\nTraining set distribution:")
print(y_train.value_counts())
print("\nTesting set distribution:")
print(y_test.value_counts())

Training instances : 6400
Testing instances  : 1600

Training set distribution:
label
Not Relevant    5264
Relevant        1136
Name: count, dtype: int64

Testing set distribution:
label
Not Relevant    1316
Relevant         284
Name: count, dtype: int64


In [ ]:
train_df = pd.DataFrame({"clean_text": X_train, "label": y_train})
test_df = pd.DataFrame({"clean_text": X_test, "label": y_test})
train_df.to_csv(f"{OUTPUT_DIR}/train_split.csv", index=False)
test_df.to_csv(f"{OUTPUT_DIR}/test_split.csv", index=False)

files.download(f"{OUTPUT_DIR}/train_split.csv")
files.download(f"{OUTPUT_DIR}/test_split.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>